In [ ]:
import json
import pandas as pd

valid_records = []
invalid_record_count = 0

with open("results/20260812_202653.records.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
            
        try:
            valid_records.append(json.loads(line))
        except json.JSONDecodeError:
            invalid_record_count += 1
            # Skip the corrupted line (usually the very last one where you killed the script)
            pass

# pd.json_normalize automatically flattens nested dictionaries.
# A JSON structure like {"result": {"score": 0.8}} becomes a column named "result.score"
df = pd.json_normalize(valid_records)

print(f"Loaded {len(df)}/{len(df) + invalid_record_count} valid records.")

In [ ]:
df.columns

In [ ]:
df['setting.model']

In [ ]:
pd.set_option('display.max_colwidth', 60)
df[["input.text", "result.translation", "result.rating", "setting.model", "setting.prompt.name"]].sort_values("result.rating")

In [ ]:
display(df.groupby(['setting.prompt.name', 'setting.model'])['metadata.elapsed_seconds'].describe())
display(df.groupby(['setting.prompt.name', 'setting.model'])['result.rating'].describe())

In [ ]:
val = df['input.text'].values
val

[v for v in val if any(k in v for k in '<{[(')]

In [ ]:
def check_citations_intact(source: str, translation: str) -> bool:
    import re
    pattern = re.compile(r'<\{\["([\w]+)"\s*,\s*("[\w:]+"|\d+)\]\}>')
    return pattern.findall(source) == pattern.findall(translation)

In [ ]:
print(check_citations_intact('See <{["Q", "112:3"]}>', 'Lihat <{["Q", "112:3"]}>'))
print(check_citations_intact('and is followed by the Successor,<{["F", 1]}>', 'dan diikuti Penerusnya,<{["F", 1]}>'))

In [ ]:
intact_citations = df.apply(lambda row: check_citations_intact(row['input.text'], row['result.translation']), axis=1)
display(intact_citations.describe())
df[~intact_citations].groupby(['setting.prompt.name', 'setting.model'])[
    ['setting.prompt.name', 'setting.model', 'input.text', 'result.translation']
].describe()

In [ ]:
df[~intact_citations]['input.text'].apply(print)
print("====")
df[~intact_citations]['result.translation'].apply(print)

In [ ]:
with pd.option_context('display.max_colwidth', None):
    display(df[["input.text", "result.translation", "result.rating", "setting.model", "setting.prompt.name"]
        ].sort_values("result.rating").tail(30))

In [ ]:
with pd.option_context('display.max_colwidth', None):
    display(df[["input.text", "result.translation", "result.rating", "setting.model", "setting.prompt.name"]
        ].sort_values("result.rating").head(30))